[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CONNECTS-SCV/bio-guides/blob/main/ab%20db/antibody_db_guide/05_humanness/05_humanness_lab.ipynb)


# 05 — humanness & humanization (Sapiens 직접 실행)

> 본문: [`05_humanness.md`](05_humanness.md) 와 **한 절씩 짝지어** 보세요.
> **전 셀 실행 16초** (실측)

**이 노트북은 도구를 직접 돌립니다.** Sapiens 언어모델을 **직접 돌려** humanness 점수·humanized 서열을 `my_run/` 에 만들어요.
각 절은 **① 직접 실행 → ② 내 결과 확인 → ③ 레퍼런스 대조** 순서예요.
내가 만든 결과는 `my_run/` 에 쌓이고, 저장소에 커밋된 `data/` 는 **대조군(레퍼런스)** 으로만 씁니다.
어느 단계를 건너뛰거나 실패해도 `resolve()` 가 `data/` 로 폴백해서 뒤 절이 계속 돌아가요.

## 0) 부트스트랩 — 저장소 클론 · 도구 설치 · 작업 폴더 이동

**Colab**: 이 셀을 그대로 실행하면 클론 → 챕터 폴더 이동 → 필요한 도구 설치까지 한 번에 끝나요.
**로컬**: 챕터 폴더 안에서 열었다면 클론 없이 진행됩니다.

In [ ]:
# ====== Colab/로컬 공용 부트스트랩 (모든 챕터 공통) ======
REPO_URL = "https://github.com/CONNECTS-SCV/bio-guides.git"   # 이 가이드 저장소 (fork 했다면 본인 주소로 바꾸세요)
CLONE_AS = "bio-guides"
CHAPTER  = "05_humanness"
PIP_PKGS = "pandas matplotlib sapiens abnumber anarci"          # 이 챕터가 실제로 돌리는 도구 (pip 이름)
NEED_HMMER = True        # ANARCI 계열은 HMMER 의 hmmscan 실행파일이 필요해요 (pip 로는 안 깔려요)
PIN_TRANSFORMERS = None    # IgFold 체크포인트가 요구하는 transformers 버전(없으면 None)

import os, sys, subprocess, pathlib, shutil, importlib.util
IN_COLAB = "google.colab" in sys.modules

# HuggingFace 가중치 다운로드가 '멈춘 채' 매달리는 일을 막습니다.
# (멈춤은 예외가 안 나서 폴백이 안 걸립니다 — 타임아웃을 걸어 실패로 바꿔야 data/ 로 이어집니다)
os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "30")   # 스트림 30초 무응답 → 끊고 재시도
os.environ.setdefault("HF_HUB_ETAG_TIMEOUT", "15")

def _run(cmd):
    print("$", cmd); subprocess.run(cmd, shell=True, check=True)

_MARK = "antibody_viz.py"           # 이 파일이 있는 폴더가 가이드 루트

def _find_root():
    """가이드 루트를 찾는다 — 챕터 폴더 안, 루트, 클론된 저장소 어디서 열어도 동작."""
    cwd = pathlib.Path.cwd()
    for base in (cwd, *list(cwd.parents)[:3]):
        if (base / _MARK).exists():
            return base
    # 클론 직후: cwd '아래'만 깊이 3까지 훑는다.
    # (상위로 올라가 rglob 하면 Colab 에서는 / 전체를 뒤지게 된다 — 매우 느리고 엉뚱한 경로를 잡는다)
    for pat in (f"*/{_MARK}", f"*/*/{_MARK}", f"*/*/*/{_MARK}"):
        hits = sorted(cwd.glob(pat))
        if hits:
            return hits[0].parent
    return None

ROOT = _find_root()
if ROOT is None and IN_COLAB:
    if not pathlib.Path(CLONE_AS).exists():
        _run(f'git clone --depth 1 "{REPO_URL}" {CLONE_AS}')
    ROOT = _find_root()
assert ROOT is not None, "repo 루트를 못 찾았습니다. 로컬이면 이 노트북을 챕터 폴더 안에서 여세요."

ADV_ROOT = ROOT.resolve()
os.chdir(ADV_ROOT / CHAPTER)        # 챕터 폴더로 이동 → data/·my_run/ 상대경로 동작
sys.path.insert(0, str(ADV_ROOT))   # antibody_viz import 보장
PY, SCRIPTS = sys.executable, ADV_ROOT / "scripts"

# --- 의존성 설치 -----------------------------------------------------------
_IMPORT = {"biopython": "Bio", "pyyaml": "yaml"}          # pip 이름 ≠ import 이름
def _have(pkg):
    mod = _IMPORT.get(pkg, pkg.split("==")[0])
    try:
        return importlib.util.find_spec(mod) is not None
    except Exception:
        return False

if NEED_HMMER and shutil.which("hmmscan") is None:
    # ANARCI 는 내부적으로 hmmscan 을 호출해요. pip install anarci 만으로는 안 깔려요.
    if IN_COLAB:
        _run("apt-get -qq update")                       # 인덱스가 낡으면 install 이 404 로 죽는다
        _run("apt-get -qq install -y hmmer")             # ← ANARCI 가 부르는 hmmscan
    else:
        print("[!] hmmscan 이 없어요 → conda install -c bioconda hmmer  (또는 apt install hmmer)")

_miss = [p for p in PIP_PKGS.split() if not _have(p)]
if _miss:
    _run(f'"{sys.executable}" -m pip -q install ' + " ".join(_miss))

if "igfold" in PIP_PKGS and importlib.util.find_spec("pkg_resources") is None:
    # setuptools 81+(2026-02) 이 pkg_resources 를 없앴는데 IgFold 의존성이 이걸 import 해요.
    _run(f'"{sys.executable}" -m pip -q install "setuptools<81"')
    importlib.invalidate_caches()

import glob as _glob
if IN_COLAB and not _glob.glob("/usr/share/fonts/**/*Nanum*", recursive=True):
    # Colab 에는 한글 폰트가 없어 그래프의 한글 라벨이 □ 로 깨집니다.
    _run("apt-get -qq update"); _run("apt-get -qq install -y fonts-nanum")


if PIN_TRANSFORMERS:
    # IgFold 체크포인트에는 옛 transformers 의 토크나이저 객체가 pickle 돼 있어요.
    # 최신 transformers(5.x) 로는 unpickle 이 실패해서(Trie/BasicTokenizer 없음) 버전을 맞춥니다.
    _ver = subprocess.run([sys.executable, "-c",
                           "import transformers;print(transformers.__version__)"],
                          capture_output=True, text=True).stdout.strip()
    if not _ver.startswith("4."):
        print(f"[transformers {_ver or 'none'} → {PIN_TRANSFORMERS}] IgFold 체크포인트 호환 버전으로 맞춥니다")
        _run(f'"{sys.executable}" -m pip -q install "transformers=={PIN_TRANSFORMERS}"')

# --- 내 산출물 폴더 & 폴백 규칙 --------------------------------------------
MYRUN = pathlib.Path("my_run"); MYRUN.mkdir(exist_ok=True)

def run_tool(args, timeout=1800):
    """도구를 서브프로세스로 실제 실행하고 출력을 셀에 그대로 보여줘요."""
    args = [str(a) for a in args]
    print("$", " ".join(args))
    try:
        p = subprocess.run(args, capture_output=True, text=True, timeout=timeout)
    except Exception as e:
        print(f"[실행 실패] {type(e).__name__}: {e}\n→ 이 단계는 건너뛰고 레퍼런스(data/)로 이어갑니다")
        return False
    out = ((p.stdout or "") + (p.stderr or "")).strip()
    print(out[-3000:] if out else "(출력 없음)")
    if p.returncode != 0:
        print(f"[실패] returncode={p.returncode} → 이 단계는 건너뛰고 레퍼런스(data/)로 이어갑니다")
    return p.returncode == 0

def resolve(name):
    """내가 방금 만든 my_run/ 결과를 우선 쓰고, 없으면 커밋된 data/ 로 폴백."""
    mine, ref = MYRUN/name, pathlib.Path("data")/name
    if mine.exists():
        print(f"[내 결과]   {mine}")
        return str(mine)
    print(f"[레퍼런스] {ref}   ← my_run 산출물이 없어 커밋본으로 이어갑니다")
    return str(ref)

print("ADV_ROOT :", ADV_ROOT)
print("작업 폴더 :", pathlib.Path.cwd(), "| Colab:", IN_COLAB)

## 1) 직접 실행 — Sapiens humanness + humanization (본문 5.1)

BioPhi CLI(`biophi sapiens`)는 **bioconda 전용**이라 Colab 에서 못 써요. 하지만 BioPhi가 내부에서 쓰는
두 부품(`sapiens` 언어모델, `abnumber` numbering)은 **둘 다 pip 에 있어요** — 그래서 같은 알고리즘을
그대로 돌릴 수 있습니다.

```bash
python scripts/sapiens_humanize.py data/demo_mab.fa \
    --scores-out my_run/demo_sapiens_scores.csv --fasta-out my_run/demo_humanized.fa
```
1. 위치별 아미노산 확률 예측 → 2. 각 위치 최대확률 아미노산으로 재구성 → 3. **원본 CDR 재이식**
(BioPhi 기본값: Kabat CDR 보존, 1 iteration)

In [ ]:
ok = run_tool([PY, SCRIPTS/"sapiens_humanize.py", "data/demo_mab.fa",
                "--scores-out", "my_run/demo_sapiens_scores.csv",
                "--fasta-out",  "my_run/demo_humanized.fa"])

## 2) 내 결과 확인 — 사슬별 humanness (본문 5.2)

In [ ]:
import pandas as pd
df = pd.read_csv(resolve("demo_sapiens_scores.csv"))
aa = list("ACDEFGHIKLMNPQRSTVWY")
df["p_input"] = [row[row["input_aa"]] if row["input_aa"] in aa else None for _, row in df.iterrows()]
hum = df.groupby("chain")["p_input"].mean()
print(hum.round(3).to_string())
print("\n[심화] Heavy 가 낮고(마우스 흔적) Light 가 높으면(사람스러움) — Ch.04 germline 결과와 일치해요.")

## 3) 내 결과 확인 — humanness 그래프 + 원본 vs humanized 변이 (본문 5.3)

In [ ]:
from antibody_viz import humanness_overview
from IPython.display import Image
png = "my_run/05_humanness_overview.png"
humanness_overview(resolve("demo_sapiens_scores.csv"), "data/demo_mab.fa",
                   resolve("demo_humanized.fa"),
                   "Humanness — Sapiens (demo mAb, 내 실행 결과)", png)
Image(png)

## 4) 레퍼런스 대조 — BioPhi CLI 결과와 같은가

`data/demo_sapiens_scores.csv`·`data/demo_humanized.fa` 는 **bioconda BioPhi CLI** 로 만들어 커밋한 대조군이에요.
pip 로 재현한 내 결과가 이것과 같은지 확인합니다(같으면 알고리즘 재현 성공).

In [ ]:
import pandas as pd, pathlib
def read_fa(p):
    d, n = {}, None
    for line in open(p):
        line = line.strip()
        if line.startswith(">"): n = line[1:].split()[0]; d[n] = ""
        elif n: d[n] += line
    return d

if pathlib.Path("my_run/demo_sapiens_scores.csv").exists():
    mine, ref = pd.read_csv("my_run/demo_sapiens_scores.csv"), pd.read_csv("data/demo_sapiens_scores.csv")
    aa = list("ACDEFGHIKLMNPQRSTVWY")
    for d in (mine, ref):
        d["p"] = [r[r["input_aa"]] for _, r in d.iterrows()]
    cmp = pd.DataFrame({"내 결과": mine.groupby("chain")["p"].mean().round(4),
                        "레퍼런스(BioPhi CLI)": ref.groupby("chain")["p"].mean().round(4)})
    cmp["차이"] = (cmp["내 결과"] - cmp["레퍼런스(BioPhi CLI)"]).abs().round(4)
    display(cmp)

    m_fa, r_fa = read_fa("my_run/demo_humanized.fa"), read_fa("data/demo_humanized.fa")
    orig = read_fa("data/demo_mab.fa")
    for sid in orig:
        nm = sum(1 for a, b in zip(orig[sid], m_fa[sid]) if a != b)
        nr = sum(1 for a, b in zip(orig[sid], r_fa[sid]) if a != b)
        print(f"{sid}: 내 humanized 변이 {nm}개 / 레퍼런스 {nr}개 | 서열 동일 = {m_fa[sid] == r_fa[sid]}")
else:
    print("my_run 산출물이 없어 대조를 건너뜁니다.")

> 다음 → 본문 [06. 구조예측 (IgFold)](../06_structure/06_structure.md)